In [1]:
from pyspark.sql import functions as F

# Bronze-Daten laden
df = spark.table(
    "bronze.synthetic_vehicle_diagnostics"
)

# 1. Anzahl Datensätze
total_rows = df.count()

# 2. Fehlende Pflichtfelder prüfen
required_columns = [
    "TestID",
    "ECU",
    "DTC",
    "ErrorCategory",
    "TestResult",
    "TestTimestamp"
]

missing_condition = F.lit(False)

for col_name in required_columns:
    missing_condition = (
        missing_condition
        | F.col(col_name).isNull()
    )

missing_rows = df.filter(missing_condition).count()

# 3. Doppelte TestIDs prüfen
duplicate_groups = (
    df.filter(F.col("TestID").isNotNull())
      .groupBy("TestID")
      .count()
      .filter(F.col("count") > 1)
)

duplicate_ids = duplicate_groups.count()

# 4. Ungültige Testergebnisse
invalid_results = df.filter(
    ~F.upper(F.trim(F.col("TestResult")))
      .isin("PASS", "FAIL")
    | F.col("TestResult").isNull()
).count()

# 5. Ungültiges DTC-Format
invalid_dtcs = df.filter(
    ~F.upper(F.trim(F.col("DTC")))
      .rlike("^[PCBU][0-9A-F]{4}$")
    | F.col("DTC").isNull()
).count()

# 6. Zulässige Fehlerkategorien
valid_categories = [
    "Short_Circuit",
    "Overtemperature",
    "Communication_Loss",
    "Voltage_Deviation",
    "Sensor_Failure",
    "Diagnostic_Error"
]

invalid_categories = df.filter(
    ~F.trim(F.col("ErrorCategory"))
      .isin(valid_categories)
    | F.col("ErrorCategory").isNull()
).count()

# 7. Gesamtergebnis
quality_status = (
    "PASS"
    if total_rows == 1000
    and missing_rows == 0
    and duplicate_ids == 0
    and invalid_results == 0
    and invalid_dtcs == 0
    and invalid_categories == 0
    else "FAIL"
)

print("=== AUTOMOTIVE DATA QUALITY REPORT ===")
print("TotalRows:", total_rows)
print("MissingRows:", missing_rows)
print("DuplicateTestIDs:", duplicate_ids)
print("InvalidResults:", invalid_results)
print("InvalidDTCs:", invalid_dtcs)
print("InvalidCategories:", invalid_categories)
print("QualityStatus:", quality_status)



StatementMeta(, 898272c9-d564-45ce-8aaf-0abd1a81764c, 3, Finished, Available, Finished, False)

=== AUTOMOTIVE DATA QUALITY REPORT ===
TotalRows: 1000
MissingRows: 0
DuplicateTestIDs: 0
InvalidResults: 0
InvalidDTCs: 0
InvalidCategories: 0
QualityStatus: PASS


In [3]:
from pyspark.sql import functions as F

# Data Quality Log vorbereiten
log_data = [(
    total_rows,
    missing_rows,
    duplicate_ids,
    invalid_results,
    invalid_dtcs,
    invalid_categories,
    quality_status
)]

log_columns = [
    "TotalRows",
    "MissingRows",
    "DuplicateTestIDs",
    "InvalidResults",
    "InvalidDTCs",
    "InvalidCategories",
    "QualityStatus"
]

df_quality_log = (
    spark.createDataFrame(log_data, log_columns)
    .withColumn("CheckedAt", F.current_timestamp())
)

# Prüfergebnisse dauerhaft speichern
(
    df_quality_log.write
    .format("delta")
    .mode("append")
    .saveAsTable("dbo.synthetic_data_quality_log")
)

print("Data Quality Log erfolgreich gespeichert!")

display(df_quality_log)


StatementMeta(, 898272c9-d564-45ce-8aaf-0abd1a81764c, 5, Finished, Available, Finished, False)

Data Quality Log erfolgreich gespeichert!


SynapseWidget(Synapse.DataFrame, 58c01def-ce56-4fbc-b63a-c3a2cab30d43)

In [2]:
# Schritt 43.3 – Automatisches Data Quality Gate

print("=== DATA QUALITY GATE ===")

if quality_status == "FAIL":
    print("Quality Gate: FAILED")
    print("Die Datenqualität ist unzureichend.")
    print("Die Pipeline-Verarbeitung wird gestoppt.")

    raise ValueError(
        "Data Quality Gate FAILED: "
        "Bronze-Daten enthalten Qualitätsprobleme."
    )

else:
    print("Quality Gate: PASSED")
    print("Die Datenqualität ist ausreichend.")
    print("Die Silver-Verarbeitung ist freigegeben.")


StatementMeta(, 898272c9-d564-45ce-8aaf-0abd1a81764c, 4, Finished, Available, Finished, False)

=== DATA QUALITY GATE ===
Quality Gate: PASSED
Die Datenqualität ist ausreichend.
Die Silver-Verarbeitung ist freigegeben.
